In [1]:
%cd ..
# %load_ext autoreload
# %autoreload 2

/home/dmatis/s/ssd/rl-optimization/turnus


In [19]:
import utils.graph_utils as graph_utils
from env import Env
import torch

graph, optimal = graph_utils.load_problem(f'data/0')

# graph.x = graph.x * -1

env = Env(graph = graph, optimal = optimal, device='cpu')

state, mask = env.reset()
observation, mask, reward, terminal, _ = env.step(1)

indices = mask.nonzero(as_tuple=True)[0]
indices[torch.randint(0, len(indices), (1,))]

tensor([4])

In [20]:
import torch
from env import Env

def random_eval(env: Env):
    state, mask = env.reset()

    terminal = False

    r_sum = 0
    while not terminal:
        indices = mask.nonzero(as_tuple=True)[0]
        action = indices[torch.randint(0, len(indices), (1,))] # pick random action
        observation, mask, reward, terminal, _ = env.step(action.item())
        r_sum += reward

    return observation, r_sum, env.vehicleID - 1


In [2]:
from env import Env

def cas_start_eval(env: Env):
    state, mask = env.reset()

    terminal = False

    while not terminal:
        action = (state.x[:, 2] + mask.logical_not() * 10000).argmin().item()
        state, mask, reward, terminal, _ = env.step(action)

    return env.vehicleID - 1

In [3]:
from env import Env

def left_to_right_eval(env: Env):
    state, mask = env.reset()

    terminal = False

    while not terminal:
        action = mask.int().argmax().item()
        state, mask, reward, terminal, _ = env.step(action)

    return env.vehicleID - 1

In [41]:
from torch_geometric.utils import degree
degree(state.edge_index[1])

tensor([ 1.,  1.,  1.,  1.,  1.,  2.,  4.,  4.,  6.,  6.,  7.,  8.,  9., 10.,
        11., 12., 13., 13., 17., 19., 20., 21., 22., 23., 24., 25., 26., 27.,
        27., 27., 28., 29., 29., 30., 32., 32., 35., 35., 36., 36., 38., 39.,
        41., 41., 43., 43., 46., 46., 48., 49., 51.], device='cuda:0')

In [4]:
import time
import torch
from env import Env
import utils.graph_utils as graph_utils

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

problems = list(range(0, 11)) + ['martin']
# problems = [0]

for problem in problems:

    graph, optimal = graph_utils.load_problem(f'data/{problem}')
    env = Env(graph = graph, optimal = optimal, device=device)

    start = time.time()
    vehicles = left_to_right_eval(env)
    took = round(time.time() - start, 2)

    gap = vehicles / optimal

    print(f'Problem {problem} ({graph.num_nodes} nodes); \t agent: {vehicles} (opt. {optimal}) - it took {took}s')

Problem 0 (6 nodes); 	 agent: 2 (opt. 2) - it took 0.01s
Problem 1 (51 nodes); 	 agent: 4 (opt. 4) - it took 0.04s
Problem 2 (79 nodes); 	 agent: 4 (opt. 4) - it took 0.06s
Problem 3 (85 nodes); 	 agent: 5 (opt. 5) - it took 0.06s
Problem 4 (107 nodes); 	 agent: 6 (opt. 6) - it took 0.08s
Problem 5 (135 nodes); 	 agent: 9 (opt. 8) - it took 0.1s
Problem 6 (162 nodes); 	 agent: 9 (opt. 9) - it took 0.12s
Problem 7 (247 nodes); 	 agent: 14 (opt. 13) - it took 0.18s
Problem 8 (417 nodes); 	 agent: 26 (opt. 26) - it took 0.32s
Problem 9 (496 nodes); 	 agent: 29 (opt. 28) - it took 0.37s
Problem 10 (929 nodes); 	 agent: 50 (opt. 49) - it took 0.75s
Problem martin (725 nodes); 	 agent: 39 (opt. 39) - it took 0.55s
